In [ ]:
import sqlite3
import requests
import json 
city = input("enter a UK city: ")  #not case sensitive 

geo_url = "https://geocoding-api.open-meteo.com/v1/search"  #geo coding url 
geo_params = {
    "name": city,
    "count": 1
}
response1 = requests.get(geo_url, params=geo_params)
response1.raise_for_status()
location_data = response1.json() 
print(json.dumps(location_data, indent=4)) #--> just for me to see the json clearly

"""extract key results from this to put into the second request : 
we know longitude and latitude is mandatory in the second request and that's what obtains weather data for the user input city """

longitude = location_data["results"][0]["longitude"]
latitude = location_data["results"][0]["latitude"]

#     *******REQUEST 2:*******

weather_url = "https://archive-api.open-meteo.com/v1/archive"       #historical data API

params = {
    "longitude" : longitude,
    "latitude"  : latitude,
    "start_date" : "2026-01-01",
    "end_date" : "2026-01-15",
    "daily" : "temperature_2m_max,temperature_2m_min,precipitation_sum",
    "timezone" : "Europe/London"
}

response = requests.get(weather_url, params=params)
response.raise_for_status()

connection = sqlite3.connect("weather.db")  #if it doesnt already exist a new DB is ceated
cursor = connection.cursor()
data = response.json()
#print(json.dumps(data, indent=4))   just for me to see the json clearly so that i can read it and obtain the variables below

#lets format this code properly to use in loops later 
dates = data["daily"]["time"]   #all the dates 
max_temp = data["daily"]["temperature_2m_max"]
min_temp = data["daily"]["temperature_2m_min"]
total_rainfall = data["daily"]["precipitation_sum"]


#table basically contains rows where each row represents one day of weather in a location
cursor.execute("DROP TABLE IF EXISTS UK_weather")
cursor.execute("""                                  
    CREATE TABLE IF NOT EXISTS UK_weather (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        city TEXT,
        date DATETIME,
        max_temperature DECIMAL,
        min_temperature DECIMAL,
        rainfall DECIMAL,
        UNIQUE(city, date)
               )
 """)

#THE UNIQUE CONSTRAINT - is so 2 rows with the same city and date dont get entered - handling duplicate rows (especially because im testing right now and rows keep getting added in)
connection.commit()

for i in range(len(dates)):
    cursor.execute(
        """
        INSERT OR IGNORE INTO UK_weather (city, date, max_temperature, min_temperature, rainfall)
        VALUES (?, ?, ?, ?, ?)
""", (
    city, dates[i], max_temp[i], min_temp[i], total_rainfall[i]
)
    )
connection.commit()

#querying the database to test out if it's working 
"""cursor.execute("SELECT * FROM UK_weather")
rows = cursor.fetchall()
for row in rows: 
    print(row)"""
#it did work btw ive just turned that bit of code into a comment for documentation purposes

#ACTUAL QUERIES For info we want

# ****************** MAX/MIN TEMPERATURE ********************
cursor.execute("""
    SELECT MAX(max_temperature)
    FROM UK_weather
""")

highest_temp = cursor.fetchone()
print("Highest temperature:", highest_temp[0])       #print(highest_temp) gives us this funny tuple (10.7,) even when it's just one val so best to index 

cursor.execute("""
    SELECT MIN(min_temperature)
    FROM UK_weather
""")
lowest_temp = cursor.fetchone()
print("Lowest temperature:", lowest_temp[0])

# ****************** TOTAL RAINFALL ********************
cursor.execute("""
    SELECT SUM(rainfall)
    FROM UK_weather
""")
total_rainfall = cursor.fetchone()
print("total rainfall:", total_rainfall[0])

cursor.execute("""
    SELECT date , rainfall
    FROM UK_weather
    WHERE rainfall > 0 
""")
rainy_days = cursor.fetchall()
print("total rainy days:", len(rainy_days))
print("here are the rainy days:")
for day in rainy_days:
    print(day)


{
    "results": [
        {
            "id": 2643743,
            "name": "London",
            "latitude": 51.50853,
            "longitude": -0.12574,
            "elevation": 25.0,
            "feature_code": "PPLC",
            "country_code": "GB",
            "admin1_id": 6269131,
            "admin2_id": 2648110,
            "timezone": "Europe/London",
            "population": 8961989,
            "country_id": 2635167,
            "country": "United Kingdom",
            "admin1": "England",
            "admin2": "Greater London"
        }
    ],
    "generationtime_ms": 0.5478859
}
Highest temperature: 10.7
Lowest temperature: -5.2
total rainfall: 58.9
total rainy days: 10
here are the rainy days:
('2026-01-02', 0.4)
('2026-01-06', 0.6)
('2026-01-07', 7.6)
('2026-01-08', 13.5)
('2026-01-09', 2.9)
('2026-01-11', 2)
('2026-01-12', 0.6)
('2026-01-13', 12.9)
('2026-01-14', 3.2)
('2026-01-15', 15.2)


prevent duplicate rows - everytime i run the code it's going to try add the rows of data back 



highest_temp = cursor.fetchone()
print("Highest temperature:", highest_temp[0])       #print(highest_temp) gives us this funny tuple (10.7,) even when it's just one val so best to index 

next steps... the code below is going to have further changes
>> no longer using user input, my program just has set list for major uk cities
>> might put everything in a loop 
>> "countryCode": "GB" #new parameter - to restrict results to just GB (ensures other country data doesnt accidentally get used)
>> alter dates to make it 5years instead of  2 weeks

In [6]:
import sqlite3
import requests
import json 
cities = ['London', 'Manchester', 'Birmingham', 'Cardiff', 'Edinburgh', 'Belfast']

#putting the connection and table outside the loop as they dont need to be looped for each city 
connection = sqlite3.connect("weather.db")  #if weather.db doesnt already exist a new DB is ceated
cursor = connection.cursor()

#Remove the existing table so that the program starts with a new set of data everytime it is run
cursor.execute("DROP TABLE IF EXISTS UK_weather")

#The table stores weather info where each row represents the weather for a particular city on a particular date
cursor.execute("""                                  
    CREATE TABLE IF NOT EXISTS UK_weather (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        city TEXT,
        date DATETIME,
        max_temperature DECIMAL,
        min_temperature DECIMAL,
        rainfall DECIMAL,
        UNIQUE(city, date)
               )
 """)
#THE UNIQUE CONSTRAINT - is so 2 rows with the same city and date dont get entered - handling duplicate rows
connection.commit()


#Collecting data from each city 
# Repeatedly call the API for weather data and insert the following data into the database
for city in cities:
    print(f"Getting weather data for {city}")

    geo_url = "https://geocoding-api.open-meteo.com/v1/search"  #geo coding url - used to convert the cities into geographical co=ordintates as that is the format expected by the Historical weather data API
    geo_params = {
        "name": city,
        "count": 1,
        "countryCode": "GB" #parameter - to restrict results to just GB so a city with the same name in a different country isn't used 
    }
    response1 = requests.get(geo_url, params=geo_params)
    response1.raise_for_status()
    location_data = response1.json() 
    # print(json.dumps(location_data, indent=4))--> I printed the response in readable format, useful in developing the program so I can see the JSON data and identify the values I needed

    #extract key results (longitude and latitude) from this API reaponse to be used in the second request (historic weather request) : 

    longitude = location_data["results"][0]["longitude"]
    latitude = location_data["results"][0]["latitude"]

    weather_url = "https://archive-api.open-meteo.com/v1/archive"       #historical weather data API

    #defined set parameters for historic weather request
    params = {
        "longitude" : longitude,
        "latitude"  : latitude,
        "start_date" : "2020-01-01",    
        "end_date" : "2026-08-01",
        "daily" : "temperature_2m_max,temperature_2m_min,precipitation_sum",
        "timezone" : "Europe/London"
    }

    response = requests.get(weather_url, params=params)
    response.raise_for_status()
    data = response.json()
    #print(json.dumps(data, indent=4)) --> I printed the JSON response clearly so that I can read and obtain the variables below

    #Extract all the weather data from the responswe 
    dates = data["daily"]["time"]   #all the dates 
    max_temp = data["daily"]["temperature_2m_max"]
    min_temp = data["daily"]["temperature_2m_min"]
    total_rainfall = data["daily"]["precipitation_sum"]

    #Store the data into the database and save the inserted records 
    for i in range(len(dates)):
        cursor.execute(
            """
            INSERT OR IGNORE INTO UK_weather (city, date, max_temperature, min_temperature, rainfall)
            VALUES (?, ?, ?, ?, ?)
    """, (
        city, dates[i], max_temp[i], min_temp[i], total_rainfall[i]
    )
        )
    connection.commit()

#ANALYSING THE DATA THROUGH QUERIES:

#Which city is the rainiest
cursor.execute("""SELECT city, SUM(rainfall) AS total_rainfall 
    FROM UK_weather
    GROUP BY city
    ORDER BY total_rainfall DESC""")

rainy_city = cursor.fetchall()
print("\nRAINIEST CITIES")

for city, total_rainfall in rainy_city:
    print(f"{city}: {total_rainfall}mm")


# Which city is the hottest 
cursor.execute("""SELECT city, AVG(max_temperature) AS avg_max_temperature
    FROM UK_weather
    GROUP BY city
    ORDER BY avg_max_temperature DESC""")

hottest_city = cursor.fetchall()
print("\nHOTTEST CITIES")

for city, temp in hottest_city:
    print(f"{city}: {temp} °C")


# For each city how many heatwave days have they had 
# By definition a heatwave is 3 or more consecutive days with a max temperature over 25°C

print("\nHeatwave days for each city:")
for city in cities:
    cursor.execute("""
        SELECT date, max_temperature
        FROM UK_weather
        WHERE city = ?
        ORDER BY date
    """, (city,))

    rows = cursor.fetchall()

    consecutive_days = 0
    heatwave_days = 0

    for date, temperature in rows:
        if temperature is not None and temperature > 25:
            consecutive_days += 1
        else:
            if consecutive_days >= 3:
                heatwave_days += consecutive_days
            consecutive_days = 0

    # Check if the final days form a heatwave
    if consecutive_days >= 3:
        heatwave_days += consecutive_days

    print(f"{city}: {heatwave_days} heatwave days")

# For each city what how many rainy days have they had
# Here, a rainy day means rainfall > 0 mm

print("\nRainy days for each city:")

cursor.execute("""
    SELECT city, COUNT(*) AS rainy_days
    FROM UK_weather
    WHERE rainfall > 0
    GROUP BY city
    ORDER BY rainy_days DESC
""")

rainy_days = cursor.fetchall()

for city, days in rainy_days:
    print(f"{city}: {days} rainy days")

Getting weather data for London
Getting weather data for Manchester
Getting weather data for Birmingham
Getting weather data for Cardiff
Getting weather data for Edinburgh
Getting weather data for Belfast

RAINIEST CITIES
Cardiff: 8861.7mm
Manchester: 7870.6mm
Belfast: 7034.7mm
Edinburgh: 6464.4mm
Birmingham: 5532.4mm
London: 4868.6mm

HOTTEST CITIES
London: 15.642577962577963 °C
Cardiff: 14.526195426195427 °C
Manchester: 14.12989604989605 °C
Birmingham: 14.052432432432433 °C
Belfast: 12.952765072765073 °C
Edinburgh: 12.526611226611227 °C

Heatwave days for each city:
London: 156 heatwave days
Manchester: 54 heatwave days
Birmingham: 84 heatwave days
Cardiff: 67 heatwave days
Edinburgh: 0 heatwave days
Belfast: 0 heatwave days

Rainy days for each city:
Belfast: 1779 rainy days
Manchester: 1768 rainy days
Edinburgh: 1720 rainy days
Cardiff: 1661 rainy days
Birmingham: 1604 rainy days
London: 1453 rainy days
